In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [2]:
#学習データ(train.csv)とテストデータ(test.csv)のデータフレームへの読み込み
train = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
test = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')

In [3]:
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

# 前処理

In [4]:
#学習データとテストデータの「male（男性）」、「female（女性）」の2つの文字列を数字の「0」「1」に数値化
train=train.replace({'male': 0 ,'female': 1 })
test=test.replace({'male': 0 ,'female': 1 })

#学習データとテストデータの「Age(年齢)」について、Pclass毎に、中央値で穴埋め
train.loc[train['Age'].isnull(), 'Age'] = train.groupby('Pclass')['Age'].transform('median')
test.loc[test['Age'].isnull(), 'Age'] = test.groupby('Pclass')['Age'].transform('median')

#学習データの「Embarked」の欠損値の２つを最頻値で穴埋め
train['Embarked'] = train['Embarked'].fillna(train['Embarked'].mode())

#テストデータの「Fare」の欠損値の１つを中央値で穴埋め
test['Fare'] = test['Fare'].fillna(test['Fare'].median())

/tmp/ipykernel_16/3498594741.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train=train.replace({'male': 0 ,'female': 1 })
/tmp/ipykernel_16/3498594741.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  test=test.replace({'male': 0 ,'female': 1 })


In [5]:
#Nameデータの2番目の領域(敬称)を抽出
train['Title_tr'] = train['Name'].map(lambda x: x.split(', ')[1].split('. ')[0])

#敬称を数値化
train['Title_tr'] = train['Title_tr'].replace(['Dr','Rev','Mlle','Major','Col','the Countess','Capt','Ms','Sir','Lady','Mme','Don','Jonkheer'], 'Others')

#少数のものをOthersに分類
train['Title_tr'] = train['Title_tr'].map( {'Master': 0 , 'Miss':1 , 'Mr':2, 'Mrs':3, 'Others':4})

#テストデータも同様に処理
test['Title_te'] = test['Name'].map(lambda x: x.split(', ')[1].split('. ')[0])
test['Title_te'] = test['Title_te'].replace(['Col','Rev','Ms','Dr','Dona'], 'Others')
test['Title_te'] = test['Title_te'].map( {'Master': 0 , 'Miss':1 , 'Mr':2, 'Mrs':3, 'Others':4})

In [6]:
#ダミー変数化 元の学習データとデータフレームを連結して、train_add_dumを作成
train_embarked_dum = pd.get_dummies(train['Embarked'], dtype='uint8')
train_add_dum = pd.concat([train,train_embarked_dum], axis=1 )
train_add_dum.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Title_tr,C,Q,S
0,1,0,3,"Braund, Mr. Owen Harris",0,22.0,1,0,A/5 21171,7.2500,NaN,S,2,0,0,1
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,38.0,1,0,PC 17599,71.2833,C85,C,3,1,0,0
2,3,1,3,"Heikkinen, Miss. Laina",1,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,1,0,0,1
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,35.0,1,0,113803,53.1000,C123,S,3,0,0,1
4,5,0,3,"Allen, Mr. William Henry",0,35.0,0,0,373450,8.0500,NaN,S,2,0,0,1


In [7]:
#テストデータも同様にダミー変数化
test_embarked_dum = pd.get_dummies(test['Embarked'], dtype='uint8')
test_add_dum = pd.concat([test,test_embarked_dum], axis=1 )
test_add_dum.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Title_te,C,Q,S
0,892,3,"Kelly, Mr. James",0,34.5,0,0,330911,7.8292,NaN,Q,2,0,1,0
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",1,47.0,1,0,363272,7.0000,NaN,S,3,0,0,1
2,894,2,"Myles, Mr. Thomas Francis",0,62.0,0,0,240276,9.6875,NaN,Q,2,0,1,0
3,895,3,"Wirz, Mr. Albert",0,27.0,0,0,315154,8.6625,NaN,S,2,0,0,1
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",1,22.0,1,1,3101298,12.2875,NaN,S,3,0,0,1


# 予測モデルの作成と予測の実施

In [8]:
from sklearn.ensemble import RandomForestClassifier
train_features = train_add_dum[['Pclass', 'Sex', 'Age', 'Fare', 'Title_tr', 'C', 'Q', 'S']].values
train_target = train['Survived'].values

# バリデーションとグリッドサーチによる分析

In [9]:
#scikit-learnのtrain_test_sprit関数をインポート
from sklearn.model_selection import train_test_split

In [10]:
#訓練用学習データと訓練用目的変数を80%、評価用学習データと評価用目的変数を20%として分割。
train_features_tr, train_features_va, train_target_tr, train_target_va = train_test_split(train_features, train_target, test_size = 0.2, random_state=0)

In [11]:
#分割された学習データと目的変数を確認
#訓練用学習データ
train_features_tr.shape

(712, 8)

In [12]:
#訓練用目的変数
train_features_va.shape

(179, 8)

In [13]:
#評価用学習データ
train_target_tr.shape

(712,)

In [14]:
#評価用目的変数
train_target_va.shape

(179,)

In [15]:
#モデルのハイパーパラメーターをチューニングするために、
#グリッドサーチ(GridSearchCV)をインポートする。
#（ここでCVは、Cross Validation(交差検証)を意味）
from sklearn.model_selection import GridSearchCV

rf_model = RandomForestClassifier(random_state=0)

#max_depthを以下の7つで試す
max_depth = [4, 5, 6, 7, 8, 9, 10]

#n_estimatorsを以下の5つで試す
n_estimators = [120, 130, 140, 150, 160]

#max_featuresを以下の3つで試す
#max_featuresは、特徴変数の最大値を意味
max_features = [6, 7, 8]

params = {'max_depth':max_depth, 'n_estimators':n_estimators, 'max_features':max_features}

#グリッドサーチで，上記ハイパーパラメータ組み合わせの7×5×3（105）通りを試す。
#入力したデータを更に5分割(cv=5)し，その内4つを訓練用データに，1つを評価用データにして、
#モデルを構築し，精度を出力することを5回繰り返す。
#5回のスコアの平均値を，モデルのスコアとする（層化5分割交差検証）。
grid_cv = GridSearchCV(estimator=rf_model, param_grid=params, cv=5)
grid_cv.fit(train_features_tr, train_target_tr)

#すべての組み合わせの内，最もスコアの高かった組み合わせと，そのスコアを表示する。
display(grid_cv.best_params_, grid_cv.best_score_,
        grid_cv.score(train_features_va, train_target_va))

{'max_depth': 8, 'max_features': 7, 'n_estimators': 130}

np.float64(0.8384615384615385)

0.8491620111731844

In [16]:
#モデルを作成の上、学習を実施 max_depth=8、n_estimators=130、max_features=7
model = RandomForestClassifier(n_estimators = 130, max_depth=8, max_features=7, random_state=0)
model.fit(train_features, train_target)

RandomForestClassifier(max_depth=8, max_features=7, n_estimators=130,
                       random_state=0)

In [17]:
#テストデータの特徴量を取得して、上記で作成したモデルで目的変数を予測
test_features = test_add_dum[['Pclass', 'Sex', 'Age', 'Fare','Title_te', 'C', 'Q', 'S']].values
predict_test_target = model.predict(test_features)

In [18]:
#予測された目的変数データと、予測されたテストデータの中身を確認
predict_test_target.shape
print(predict_test_target)

[0 0 0 0 1 0 0 0 1 0 0 0 1 0 1 1 0 0 1 0 0 1 1 0 1 0 1 0 0 0 0 0 1 0 1 0 0
 0 0 0 0 1 0 1 1 0 0 0 1 1 0 0 1 1 0 0 0 0 0 1 0 0 0 1 1 0 1 0 0 1 1 0 0 0
 1 0 0 1 0 1 1 0 0 0 0 0 1 0 1 1 1 0 1 0 0 0 1 0 1 0 1 0 0 0 1 0 0 0 0 0 0
 1 1 1 1 0 0 1 0 1 1 0 1 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 0 0
 1 0 1 0 0 1 0 0 1 1 1 1 1 1 1 0 0 0 0 0 1 0 0 0 0 0 0 1 1 0 1 1 0 0 1 0 1
 0 1 0 0 0 0 0 1 0 1 0 1 1 0 0 1 1 0 1 0 0 0 0 1 0 0 0 0 1 0 0 1 0 1 0 1 0
 1 0 1 0 0 1 0 0 0 1 0 0 0 0 0 0 1 1 1 1 0 0 0 0 1 0 1 0 1 0 0 0 0 0 0 0 1
 0 0 0 1 1 0 0 0 0 0 0 0 0 1 1 0 1 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0
 1 0 0 0 0 0 0 0 1 1 0 1 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 1 1 0 1 0 0 0 1 0 0
 1 0 0 0 0 0 1 0 0 0 1 1 1 0 0 0 1 1 0 0 0 1 0 1 0 0 0 0 1 1 0 1 0 0 0 1 0
 0 1 0 0 1 1 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 1 1 0 0 1 0 1 0 0 1 0 1 0 0 0 0
 0 1 1 1 1 0 0 1 0 0 1]


In [19]:
submission = pd.DataFrame({'PassengerId': test['PassengerId'], 'Survived': predict_test_target})
submission.to_csv('submission_RandomForestClassifier_2.csv', index = False )